<a href="https://colab.research.google.com/github/OJB-Quantum/Monte-Carlo-Sim/blob/main/CUDA_Accelerated_GDS_OASIS_to_PEC_OPC_in_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)

In [ ]:
"""CUDA-Accelerated Layout Generation: Auto-Scaling Domain.

This script performs 193i Immersion DUV OPC and E-beam PEC on GDSII/OASIS files.
It features AUTOMATIC DOMAIN SCALING to ensure the entire design layout fits
within the simulation viewport, regardless of physical size.

Usage:
    Run in a Google Colab environment with a GPU runtime.
    Upload a .gds or .oas file when prompted.
"""
!nvidia-smi

Tue Jan 13 03:44:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             11W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import sys
import subprocess
from dataclasses import dataclass, replace
from typing import Tuple, List

# --- Environment Setup (Colab Specifics) ---
def _install_dependencies() -> None:
    """Check for CUDA and install necessary system/Python libraries."""
    packages = ["gdstk", "cupy-cuda12x", "scikit-image"]
    try:
        import gdstk  # noqa: F401
        import cupy   # noqa: F401
        print("Dependencies are already installed.")
    except ImportError:
        print("Installing dependencies (gdstk, cupy, skimage)...")
        try:
            out = subprocess.check_output(["nvidia-smi"], text=True)
            if "CUDA Version: 11" in out:
                packages = [p.replace("12x", "11x") for p in packages]
        except Exception:
            pass
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
        subprocess.check_call(cmd)
        print("Installation complete.")

_install_dependencies()

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import cupy as cp
import cupyx.scipy.special as cpx_special
import gdstk
from skimage.draw import polygon
from google.colab import files

# --- Configuration & Control Knobs ---

mpl.rcParams.update({
    "figure.dpi": 600,
    "image.cmap": "inferno",
})

@dataclass(frozen=True)
class SimConfig:
    """Simulation configuration parameters (Control Knobs)."""

    # --- Polarity Control Knob ---
    # Toggle this to invert your mask!
    # "Positive": Polygons are 1 (Bright), Background is 0 (Dark)
    # "Negative": Polygons are 0 (Dark), Background is 1 (Bright)
    mask_polarity: str = "Positive (GDS=1)" #@param ["Positive (GDS=1)", "Negative (GDS=0)"]

    # --- Domain Setup ---
    # Note: The script will AUTO-SCALE this to fit your full chip.
    domain_width_um: float = 10.0

    # Grid Resolution: High resolution for large chips
    grid_n: int = 4096

    # --- DUV Immersion Optical Model (193i) ---
    opt_wavelength_nm: float = 193.0
    opt_na: float = 1.35
    opt_threshold: float = 0.35
    opt_photons_per_pixel: float = 2000.0

    # OPC Optimization
    opt_opc_iters: int = 40
    opt_opc_step: float = 0.80
    opt_sigmoid_slope: float = 0.05

    # --- E-beam Lithography (PEC) ---
    eb_sigma_f_nm: float = 8.0
    eb_sigma_b_nm: float = 140.0
    eb_eta: float = 0.40
    eb_threshold: float = 0.50

    # PEC Optimization
    eb_reg_lambda: float = 5.0e-3
    eb_dose_clip: float = 6.0

    # --- System ---
    mc_electrons_total: int = 5_000_000
    mc_chunk_size: int = 1_000_000
    seed: int = 42

INITIAL_CFG = SimConfig()

# --- Utility Functions ---

def domain_width_nm(cfg: SimConfig) -> float:
    return float(cfg.domain_width_um) * 1000.0

def pixel_nm(cfg: SimConfig) -> float:
    return domain_width_nm(cfg) / float(cfg.grid_n)

def airy_disk_psf(shape: Tuple[int, int], cfg: SimConfig, px_size_nm: float) -> cp.ndarray:
    ny, nx = shape
    y = cp.arange(ny) - ny // 2
    x = cp.arange(nx) - nx // 2
    yy, xx = cp.meshgrid(y, x, indexing="ij")
    r_nm = cp.sqrt(xx**2 + yy**2) * px_size_nm
    k0 = 2.0 * np.pi / cfg.opt_wavelength_nm
    v = k0 * cfg.opt_na * r_nm
    epsilon = 1e-8
    v = cp.maximum(v, epsilon)
    jinc = 2.0 * cpx_special.j1(v) / v
    psf = jinc ** 2
    return (psf / cp.sum(psf)).astype(cp.float32)

def double_gaussian_psf(shape: Tuple[int, int], sigma_f_px: float, sigma_b_px: float, eta: float) -> cp.ndarray:
    ny, nx = shape
    y = cp.arange(ny) - ny // 2
    x = cp.arange(nx) - nx // 2
    yy, xx = cp.meshgrid(y, x, indexing="ij")
    r2 = xx**2 + yy**2
    psf_f = cp.exp(-r2 / (2.0 * sigma_f_px**2))
    psf_b = cp.exp(-r2 / (2.0 * sigma_b_px**2))
    psf_f /= cp.sum(psf_f)
    psf_b /= cp.sum(psf_b)
    psf = (1.0 - eta) * psf_f + eta * psf_b
    return psf.astype(cp.float32)

def fft_convolve2d(img: cp.ndarray, psf_f: cp.ndarray) -> cp.ndarray:
    out = cp.fft.ifft2(cp.fft.fft2(img) * psf_f)
    return cp.real(out).astype(cp.float32)

# --- Layout Processor (Robust Scaling & Polarity) ---

def load_analyze_rasterize(cfg: SimConfig) -> Tuple[cp.ndarray, SimConfig]:
    """Load GDS/OASIS, Auto-Scale, and Apply Polarity."""
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")

    filename = next(iter(uploaded.keys()))
    ext = os.path.splitext(filename)[1].lower().strip()
    print(f"\nProcessing {filename}...")

    # 1. Read Library
    try:
        if ext == ".gds":
            lib = gdstk.read_gds(filename)
        elif ext == ".oas":
            lib = gdstk.read_oas(filename)
        else:
            raise ValueError(f"Unsupported: {ext}")

        top_cells = lib.top_level()
        if not top_cells:
            print("No top cell. Searching...")
            cell = max(lib.cells, key=lambda c: len(c.polygons) + len(c.references))
        else:
            cell = top_cells[0]
        print(f"Top Cell: '{cell.name}'")

        # 2. Robust Bounding Box
        bbox = cell.bounding_box()
        if bbox is None:
            raise RuntimeError("Layout is empty.")

        min_x, min_y = bbox[0]
        max_x, max_y = bbox[1]

        design_w = max_x - min_x
        design_h = max_y - min_y
        center_x = (min_x + max_x) / 2.0
        center_y = (min_y + max_y) / 2.0

        print(f"Detected Size: {design_w:.2f} x {design_h:.2f} um")

        # 3. Auto-Scale Viewport
        max_dim = max(design_w, design_h)
        new_width_um = max(cfg.domain_width_um, max_dim * 1.05)

        print(f"Auto-Scaled Viewport: {new_width_um:.2f} um")

        pixel_size_nm = (new_width_um * 1000.0) / cfg.grid_n
        if pixel_size_nm > 200.0:
            print(f"⚠️ Resolution Warning: {pixel_size_nm:.2f} nm/px (Preview Mode)")

        new_cfg = replace(cfg, domain_width_um=new_width_um)

        # 4. Rasterize (Safe Loop)
        print("Rasterizing...")
        cell.flatten()
        polys = cell.get_polygons(apply_repetitions=True, include_paths=True)

        mask = np.zeros((new_cfg.grid_n, new_cfg.grid_n), dtype=np.float32)
        scale = new_cfg.grid_n / new_cfg.domain_width_um
        origin_x = center_x - (new_cfg.domain_width_um / 2.0)
        origin_y = center_y - (new_cfg.domain_width_um / 2.0)

        count = 0
        for p in polys:
            # FIX: Handle gdstk.Polygon object vs numpy array
            if hasattr(p, 'points'):
                pts = p.points
            else:
                pts = p

            pts = np.asarray(pts)
            if pts.shape[0] < 3: continue

            # Map physical coords to pixel coords
            px_coords = (pts - [origin_x, origin_y]) * scale
            rr, cc = polygon(px_coords[:, 1], px_coords[:, 0], shape=mask.shape)
            mask[rr, cc] = 1.0
            count += 1

        print(f"Rasterized {count} polygons.")

        # 5. Apply Polarity Control Knob
        if "Negative" in new_cfg.mask_polarity:
            print(f"Applying Negative Polarity (Inverting 0 <-> 1)...")
            mask = 1.0 - mask
        else:
            print(f"Applying Positive Polarity (Standard)...")

        return cp.asarray(mask), new_cfg

    except Exception as e:
        print(f"ERROR: {e}")
        return cp.zeros((cfg.grid_n, cfg.grid_n), dtype=cp.float32), cfg

# --- Simulation Logic ---

def run_optical_correction(target: cp.ndarray, cfg: SimConfig, psf_f: cp.ndarray) -> Tuple[cp.ndarray, cp.ndarray]:
    print("Running 193i OPC...")
    mask = target.copy()
    rng = cp.random.default_rng(cfg.seed)
    mask += 0.01 * rng.standard_normal(mask.shape, dtype=cp.float32)
    mask = cp.clip(mask, 0.0, 1.0)

    for _ in range(cfg.opt_opc_iters):
        aerial = fft_convolve2d(mask, psf_f)
        diff = (aerial - cfg.opt_threshold) / cfg.opt_sigmoid_slope
        printed_soft = 1.0 / (1.0 + cp.exp(-diff))
        error = printed_soft - target
        grad = error * (printed_soft * (1.0 - printed_soft)) / cfg.opt_sigmoid_slope
        grad_mask = fft_convolve2d(grad, psf_f)
        mask = cp.clip(mask - cfg.opt_opc_step * grad_mask, 0.0, 1.0)

    expected = fft_convolve2d(mask, psf_f)
    lam = cp.clip(expected * cfg.opt_photons_per_pixel, 0.0, None)
    noisy = rng.poisson(lam).astype(cp.float32) / cfg.opt_photons_per_pixel
    return mask, noisy

def run_ebeam_correction(target: cp.ndarray, cfg: SimConfig, psf_f: cp.ndarray, sigma_f: float, sigma_b: float) -> Tuple[cp.ndarray, cp.ndarray]:
    print("Running E-beam PEC...")
    target_f = cp.fft.fft2(target)
    denom = (cp.abs(psf_f)**2) + cfg.eb_reg_lambda
    dose_f = target_f * cp.conj(psf_f) / denom
    dose = cp.real(cp.fft.ifft2(dose_f))
    dose = cp.clip(dose, 0.0, cfg.eb_dose_clip)

    print("Running Monte Carlo...")
    dose_flat = dose.ravel()
    dose_sum = float(cp.sum(dose_flat))
    if dose_sum == 0: return dose, cp.zeros_like(dose)

    cdf = cp.cumsum(dose_flat)
    cdf_max = cdf[-1]
    rng = cp.random.default_rng(cfg.seed)
    energy_acc = cp.zeros(dose.size, dtype=cp.int32)
    n_processed = 0
    while n_processed < cfg.mc_electrons_total:
        batch = min(cfg.mc_chunk_size, cfg.mc_electrons_total - n_processed)
        u = rng.random(batch, dtype=cp.float32) * cdf_max
        indices = cp.searchsorted(cdf, u).astype(cp.int32)
        y0 = indices // cfg.grid_n
        x0 = indices % cfg.grid_n
        is_back = rng.random(batch) < cfg.eb_eta
        sigmas = cp.where(is_back, sigma_b, sigma_f)
        dy = rng.standard_normal(batch) * sigmas
        dx = rng.standard_normal(batch) * sigmas
        y_final = cp.clip(y0 + cp.rint(dy).astype(cp.int32), 0, cfg.grid_n - 1)
        x_final = cp.clip(x0 + cp.rint(dx).astype(cp.int32), 0, cfg.grid_n - 1)
        flat_indices = y_final * cfg.grid_n + x_final
        cp.add.at(energy_acc, flat_indices, 1)
        n_processed += batch

    energy = energy_acc.astype(cp.float32) / cfg.mc_electrons_total * dose_sum
    energy = energy.reshape((cfg.grid_n, cfg.grid_n))
    return dose, energy

# --- Main Execution ---

target_gpu, UPDATED_CFG = load_analyze_rasterize(INITIAL_CFG)
px_size = pixel_nm(UPDATED_CFG)

# Setup PSFs
psf_opt = airy_disk_psf((UPDATED_CFG.grid_n, UPDATED_CFG.grid_n), UPDATED_CFG, px_size)
psf_opt_f = cp.fft.fft2(cp.fft.ifftshift(psf_opt))

psf_eb = double_gaussian_psf(
    (UPDATED_CFG.grid_n, UPDATED_CFG.grid_n),
    UPDATED_CFG.eb_sigma_f_nm / px_size,
    UPDATED_CFG.eb_sigma_b_nm / px_size,
    UPDATED_CFG.eb_eta
)
psf_eb_f = cp.fft.fft2(cp.fft.ifftshift(psf_eb))

# Execute
opc_mask, opc_res = run_optical_correction(target_gpu, UPDATED_CFG, psf_opt_f)
opt_unc_aer = fft_convolve2d(target_gpu, psf_opt_f)
rng = cp.random.default_rng(UPDATED_CFG.seed)
lam = cp.clip(opt_unc_aer * UPDATED_CFG.opt_photons_per_pixel, 0.0, None)
opt_unc_res = rng.poisson(lam).astype(cp.float32) / UPDATED_CFG.opt_photons_per_pixel

pec_dose, pec_res = run_ebeam_correction(
    target_gpu, UPDATED_CFG, psf_eb_f,
    UPDATED_CFG.eb_sigma_f_nm / px_size, UPDATED_CFG.eb_sigma_b_nm / px_size
)
_, eb_unc_res = run_ebeam_correction(
    target_gpu, UPDATED_CFG, psf_eb_f,
    UPDATED_CFG.eb_sigma_f_nm / px_size, UPDATED_CFG.eb_sigma_b_nm / px_size
)

# Plot
def plot_results(target, opc_mask, opt_corr, opt_unc, pec_dose, eb_corr, eb_unc, cfg):
    c_target = cp.asnumpy(target)
    c_opc_mask = cp.asnumpy(opc_mask)
    c_opt_corr = cp.asnumpy(opt_corr)
    c_opt_unc = cp.asnumpy(opt_unc)
    c_pec_dose = cp.asnumpy(pec_dose)
    c_eb_corr = cp.asnumpy(eb_corr)
    c_eb_unc = cp.asnumpy(eb_unc)

    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 0.8])
    width_nm = domain_width_nm(cfg)
    extent = [-width_nm/2, width_nm/2, -width_nm/2, width_nm/2]

    # FIX: Added origin='lower' to correct upside-down plots
    def show_img(ax, data, title, cmap='gray', cb=False):
        im = ax.imshow(data, extent=extent, cmap=cmap, origin='lower')
        ax.set_title(title)
        ax.set_xlabel("x (nm)")
        ax.set_ylabel("y (nm)")
        if cb: plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    show_img(fig.add_subplot(gs[0, 0]), c_target, f"1. Target ({cfg.domain_width_um/1000:.1f} mm)")
    show_img(fig.add_subplot(gs[0, 1]), c_opc_mask, "2. OPC Mask", 'magma', True)
    show_img(fig.add_subplot(gs[0, 2]), c_opt_corr, "3. Aerial Image", 'inferno', True)
    show_img(fig.add_subplot(gs[0, 3]), (c_opt_corr >= cfg.opt_threshold), "4. Developed")

    show_img(fig.add_subplot(gs[1, 0]), c_target, "1. Target")
    show_img(fig.add_subplot(gs[1, 1]), c_pec_dose, "2. PEC Dose", 'viridis', True)
    show_img(fig.add_subplot(gs[1, 2]), c_eb_corr, "3. Deposited Energy", 'inferno', True)
    show_img(fig.add_subplot(gs[1, 3]), (c_eb_corr >= cfg.eb_threshold), "4. Developed")

    mid = cfg.grid_n // 2
    x = (np.arange(cfg.grid_n) - mid) * pixel_nm(cfg)

    ax9 = fig.add_subplot(gs[2, :2])
    ax9.plot(x, c_target[mid], 'k--', label="Target")
    ax9.plot(x, c_opt_unc[mid], 'gray', alpha=0.6, label="Uncorrected")
    ax9.plot(x, c_opt_corr[mid], '#D32F2F', lw=2, label="OPC")
    ax9.axhline(cfg.opt_threshold, color='g', ls=':', label="Threshold")
    ax9.legend(); ax9.set_title("Optical Cross-Section")

    ax10 = fig.add_subplot(gs[2, 2:])
    ax10.plot(x, c_target[mid], 'k--', label="Target")
    ax10.plot(x, c_eb_unc[mid], 'gray', alpha=0.6, label="Uncorrected")
    ax10.plot(x, c_eb_corr[mid], '#1976D2', lw=2, label="PEC")
    ax10.axhline(cfg.eb_threshold, color='g', ls=':', label="Threshold")
    ax10.legend(); ax10.set_title("E-beam Cross-Section")
    plt.tight_layout()
    plt.show()

plot_results(target_gpu, opc_mask, opc_res, opt_unc_res, pec_dose, pec_res, eb_unc_res, UPDATED_CFG)

Dependencies are already installed.


Saving Onri_B_CPW_DW_OASIS.oas to Onri_B_CPW_DW_OASIS (1).oas

Processing Onri_B_CPW_DW_OASIS (1).oas...
Top Cell: 'TOP'
Detected Size: 4752.38 x 3179.02 um
Auto-Scaled Viewport: 4990.00 um
⚠️ Resolution Warning: 1218.26 nm/px (Preview Mode)
Rasterizing...
Rasterized 43448 polygons.
Applying Positive Polarity (Standard)...
Running 193i OPC...
Running E-beam PEC...
Running Monte Carlo...
Running E-beam PEC...
Running Monte Carlo...
